In [ ]:
import os
import glob
import pandas as pd
import re

In [6]:
# Rutas de origen (data/raw/pyeph/.db/) y destino (data/raw/)
base_dir = os.path.join("..", "data", "raw")
pyeph_db_dir = os.path.join(base_dir, "pyeph", ".db")
output_dir = os.path.join("..", "data", "processed")

os.makedirs(output_dir, exist_ok=True)


In [ ]:
# 1. DEFINE LAS COLUMNAS QUE QUIERES ELIMINAR (LISTAS DE DROP)
DROP_COLS_INDIVIDUAL = [
    'IMPUTA', 'PP02C1', 'PP02C2', 'PP02E', 'PP02H', 'PP02I'
]

DROP_COLS_HOGAR = [
    'REALIZADA', 'ENTREVISTADOR', 'PROPOND'
]

# Variables de unión (Se protegen para evitar eliminarlas accidentalmente)
COLS_LLAVE = ['CODUSU', 'NRO_HOGAR', 'ANO4', 'TRIMESTRE']

# Función auxiliar para leer un CSV/TXT de un ZIP sin descomprimir a disco
def leer_df_desde_zip(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as z:
        # Busca el primer archivo de datos dentro del zip (.csv o .txt)
        archivos_internos = [f for f in z.namelist() if f.endswith('.csv') or f.endswith('.txt')]
        if not archivos_internos:
            raise FileNotFoundError(f"No se encontró un CSV o TXT dentro de {zip_path}")
        
        with z.open(archivos_internos[0]) as f:
            return pd.read_csv(f, sep=';', low_memory=False)

# 2. BUSCAR ARCHIVOS INDIVIDUALES CON EL PATRÓN ESPECÍFICO
patron_ind = os.path.join(pyeph_db_dir, "individual", "base_individual_*.zip")
archivos_ind = glob.glob(patron_ind)

for file_ind in archivos_ind:
    nombre_archivo_ind = os.path.basename(file_ind)
    
    # Extraer el sufijo del período (ejemplo: '2023T3' de 'base_individual_2023T3.zip')
    match = re.search(r"base_individual_(\d{4}T\d)\.zip", nombre_archivo_ind)
    if not match:
        continue
        
    periodo_str = match.group(1) # 'AAAATN'
    
    # Construir la ruta equivalente para la base de hogar
    file_hog = os.path.join(pyeph_db_dir, "hogar", f"base_hogar_{periodo_str}.zip")
    
    if not os.path.exists(file_hog):
        print(f"⚠️ Par omitido (no existe base_hogar_{periodo_str}.zip en hogar): {nombre_archivo_ind}")
        continue

    parquet_path = os.path.join(output_dir, f"eph_merged_{periodo_str}.parquet")
    
    if os.path.exists(parquet_path):
        print(f"⏩ Ya existe: {parquet_path}")
        continue
        
    try:
        # Cargar DataFrames leyendo directamente desde los .zip
        df_ind = leer_df_desde_zip(file_ind)
        df_hog = leer_df_desde_zip(file_hog)

        # Filtrar listas de drop (resguardando las llaves)
        drop_ind = [c for c in DROP_COLS_INDIVIDUAL if c in df_ind.columns and c not in COLS_LLAVE]
        drop_hog = [c for c in DROP_COLS_HOGAR if c in df_hog.columns and c not in COLS_LLAVE]

        # Aplicar el drop
        df_ind = df_ind.drop(columns=drop_ind)
        df_hog = df_hog.drop(columns=drop_hog)

        # 3. MERGE 1 a N (Individual <- Hogar)
        df_merged = pd.merge(
            df_ind,
            df_hog,
            on=['CODUSU', 'NRO_HOGAR', 'ANO4', 'TRIMESTRE'],
            how='left',
            suffixes=('', '_hogar')
        )

        # OPTIMIZACIÓN DE TIPOS DE DATOS Y RAM
        for col in df_merged.select_dtypes(include=['object', 'category']).columns:
            df_merged[col] = df_merged[col].astype(str).replace({'nan': None, 'None': None})

        floats = df_merged.select_dtypes(include=['float64']).columns
        df_merged[floats] = df_merged[floats].astype('float32')
        
        ints = df_merged.select_dtypes(include=['int64']).columns
        df_merged[ints] = df_merged[ints].astype('int32')

        # 4. GUARDAR EN PARQUET
        df_merged.to_parquet(parquet_path, index=False, engine='pyarrow', compression='snappy')
        print(f"✔ Procesado y guardado: {os.path.basename(parquet_path)} ({len(df_merged)} filas)")

    except Exception as e:
        print(f"✖ Error al procesar {nombre_archivo_ind}: {e}")

,CODUSU,ANO4,TRIMESTRE,NRO_HOGAR,REALIZADA,REGION,MAS_500,AGLOMERADO,PONDERA,IV1,...,GDECCFR,PDECCFR,ADECCFR,PONDIH,VII1_1,VII1_2,VII2_1,VII2_2,VII2_3,VII2_4
1,TQRMNOQUXHKOKMCDEGKDB00475140,2016,2,1,1,43,N,14,77,1,...,NaN,12.0,12.0,0,2,0,98,0,0,0
2,TQRMNOQQYHMMKTCDEGKDB00475009,2016,2,1,1,43,N,14,66,1,...,NaN,7.0,7.0,79,2,0,98,0,0,0
3,TQRMNORVUHJMKSCDEGKDB00475326,2016,2,1,1,43,N,14,59,1,...,NaN,7.0,8.0,66,1,0,2,0,0,0
4,TQRMNOPQUHMMKTCDEGKDB00475005,2016,2,1,1,43,N,14,66,1,...,NaN,9.0,10.0,81,2,0,98,0,0,0
5,TQRMNORVRHJMKSCDEGKDB00475324,2016,2,1,1,43,N,14,59,1,...,NaN,3.0,4.0,65,2,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18792,TQRMNORXVHMNKTCDEOHCH00487178,2016,2,1,1,44,N,91,77,1,...,NaN,6.0,5.0,89,1,0,97,0,0,0
18793,TQRMNORXWHMNKTCDEOHCH00487179,2016,2,1,1,44,N,91,77,1,...,NaN,9.0,8.0,90,1,0,98,0,0,0
18794,TQRMNOQRPHLMKMCDEIOAH00486340,2016,2,1,1,43,N,38,89,1,...,NaN,12.0,12.0,0,1,0,2,0,0,0
18795,TQRMNOQQUHKOLLCDEIOAH00486313,2016,2,1,1,43,N,38,117,1,...,NaN,8.0,9.0,135,2,0,98,0,0,0
